In [1]:
import ase
import numpy as np
import pyscf
import time
import os
from pyscf import gto, dft
from pyscf.scf import hf
hf.MUTE_CHKFILE = True
import equiv_dens.utils.base as utils

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
# convert data
data = np.load('datasets/ethanol_dft_train.npy', allow_pickle=True).item()
print(data['atom_numbers'])
print(data['atom_types'])
data['atom_numbers'][2] = 16
data['atom_types'][2] = 'S'
np.save('datasets/ethanethiol_dft_train.npy', data, allow_pickle=True)
print(data['atom_numbers'])
print(data['atom_types'])

data = np.load('datasets/ethanol_dft_valid.npy', allow_pickle=True).item()
print(data['atom_numbers'])
print(data['atom_types'])
data['atom_numbers'][2] = 16
data['atom_types'][2] = 'S'
np.save('datasets/ethanethiol_dft_valid.npy', data, allow_pickle=True)
print(data['atom_numbers'])
print(data['atom_types'])

data = np.load('datasets/ethanol_dft_test.npy', allow_pickle=True).item()
print(data['atom_numbers'])
print(data['atom_types'])
data['atom_numbers'][2] = 16
data['atom_types'][2] = 'S'
np.save('datasets/ethanethiol_dft_test.npy', data, allow_pickle=True)
print(data['atom_numbers'])
print(data['atom_types'])

[6 6 8 1 1 1 1 1 1]
['C' 'C' 'O' 'H' 'H' 'H' 'H' 'H' 'H']
[ 6  6 16  1  1  1  1  1  1]
['C' 'C' 'S' 'H' 'H' 'H' 'H' 'H' 'H']
[6 6 8 1 1 1 1 1 1]
['C' 'C' 'O' 'H' 'H' 'H' 'H' 'H' 'H']
[ 6  6 16  1  1  1  1  1  1]
['C' 'C' 'S' 'H' 'H' 'H' 'H' 'H' 'H']
[6 6 8 1 1 1 1 1 1]
['C' 'C' 'O' 'H' 'H' 'H' 'H' 'H' 'H']
[ 6  6 16  1  1  1  1  1  1]
['C' 'C' 'S' 'H' 'H' 'H' 'H' 'H' 'H']


In [10]:
# load data
data = np.load('datasets/ethanethiol_dft_train.npy', allow_pickle=True).item()

In [11]:
# test different basis sets
basis_sets = ['631gs', '631gss', 'def2svp', 'ccpvdz', 'aug-ccpvdz']
atom_pos = data['positions']
atom_types = data['atom_numbers']

for basis in basis_sets:
    print('basis', basis)
    start = time.time()
    pos = atom_pos[0]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)

basis 631gs
converged SCF energy = -477.408024029497
--------------- RKS gradients ---------------
         x                y                z
0 C     0.3407866899     0.1632287393    -0.1905619970
1 C    -0.0358250721    -0.0127920097     0.0011298902
2 S    -0.0175221932    -0.4958834259     0.2563150681
3 H     0.0071070260     0.0082819821    -0.0139925646
4 H     0.0230866811     0.0086796591    -0.0026022151
5 H     0.0161050823     0.0087313099    -0.0234852946
6 H     0.0345117432    -0.0008948695    -0.0114465183
7 H     0.0070696721    -0.0245253938     0.0129918259
8 H    -0.3753101158     0.3451846588    -0.0283492499
----------------------------------------------
elapsed 5.377262353897095
basis 631gss
converged SCF energy = -477.422014218138
--------------- RKS gradients ---------------
         x                y                z
0 C     0.3407896848     0.1621220720    -0.1904980180
1 C    -0.0348601404    -0.0129995629     0.0001566653
2 S    -0.0262878759    -0.485317

In [9]:

# load data
data = np.load('datasets/ethanethiol_dft_test.npy', allow_pickle=True).item()

In [10]:
atom_pos = data['positions']
atom_types = data['atom_numbers']
save_path = 'datasets/ethanethiol_pyscf_augccpvdz_dft_test.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(atom_pos)):
    print('calc', i)
    start = time.time()
    pos = atom_pos[i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='augccpvdz')
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
    res.append(calc_dict)
    results.append(res)
    
    if i%10 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)

results len 0
calc 0
converged SCF energy = -477.465845200118
--------------- RKS gradients ---------------
         x                y                z
0 C     0.1841750131     0.1389573077     0.1665203266
1 C     0.0033871801     0.0149773898    -0.0086589163
2 S    -0.4457808493    -0.1864295364     0.3465522415
3 H     0.0204762781     0.0233567648     0.0090301783
4 H    -0.0000206086     0.0119189177     0.0461442670
5 H     0.0108429268     0.0104841831     0.0072461027
6 H    -0.0117707098    -0.0079834494    -0.0056890021
7 H     0.0167274716    -0.0101496426    -0.0021317911
8 H     0.2219674599     0.0048813117    -0.5590035789
----------------------------------------------
elapsed 16.050573587417603
mo occ [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 

In [4]:
np_data = np.load('datasets/ethanethiol_dft_test.npy', allow_pickle=True).item()

load_path = 'datasets/ethanethiol_pyscf_augccpvdz_dft_test.npy'
pyscf_data = np.load(load_path, allow_pickle=True)
np_data['energy'] = []
np_data['forces'] = []
for calc in pyscf_data:
    np_data['energy'].append(calc[1]['energy'])
    np_data['forces'].append(-calc[1]['forces'] * utils.to_bohr)
    
np_data['energy'] = np.array(np_data['energy'])
np_data['forces'] = np.array(np_data['forces'])
print('energy shape', np_data['energy'].shape)
print('forces shape', np_data['forces'].shape)
save_path = 'datasets/ethanethiol_augccpvdz_test.npy'
np.save(save_path, np_data, allow_pickle=True)

energy shape (1000,)
forces shape (1000, 9, 3)


In [ ]:
s